# 01 — Data Exploration

**Goal:** Pull all FRED series, verify shapes, date ranges, and confirm 
the series IDs in `preprocessing_config.yaml` are active and returning data.

**Outputs:**
- Summary table: series_id, start_date, end_date, obs_count, latest_value
- Quick time series plots for each series
- Identify any gaps, discontinuations, or surprises

**Prerequisites:** Set `FRED_API_KEY` in your `.env` file.

In [ ]:
import os
import sys
from pathlib import Path

# Add backend to path so we can import project modules
sys.path.insert(0, str(Path.cwd().parent / 'backend'))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / 'backend' / '.env')

import pandas as pd
import matplotlib.pyplot as plt
import yaml

from data.ingestion import load_series_config, generate_dummy_series

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 10

In [ ]:
# Load series config
config = load_series_config()
print(f'{len(config)} series configured')
for s in config:
    print(f"  {s['id']:40s} {s['frequency']:10s} role={s['role']}")

In [ ]:
# Option A: Fetch from FRED (requires API key)
# from data.ingestion import fetch_all_series
# raw = fetch_all_series()

# Option B: Use dummy data for development
raw = generate_dummy_series()
print(f'Loaded {len(raw)} series')

In [ ]:
# Summary table
summary = []
for sid, df in raw.items():
    summary.append({
        'series_id': sid,
        'start': df['date'].min().strftime('%Y-%m'),
        'end': df['date'].max().strftime('%Y-%m'),
        'obs_count': len(df),
        'latest_value': round(df['value'].iloc[-1], 2),
    })

pd.DataFrame(summary).set_index('series_id')

In [ ]:
# Quick plots
fig, axes = plt.subplots(len(raw), 1, figsize=(12, 3 * len(raw)), sharex=False)
if len(raw) == 1:
    axes = [axes]

for ax, (sid, df) in zip(axes, raw.items()):
    ax.plot(df['date'], df['value'], linewidth=1)
    ax.set_title(sid, fontsize=10)
    ax.axvspan('2020-03-01', '2021-09-01', alpha=0.1, color='red', label='COVID')

plt.tight_layout()
plt.show()

## Next Steps

- [ ] Confirm all series IDs return data from FRED
- [ ] Note any series with gaps or unexpected date ranges
- [ ] Proceed to `02_stationarity_analysis.ipynb`